# 04 · Limpieza, transformación y almacenamiento en Parquet (ETL)

**Objetivos de aprendizaje**

- Detectar y eliminar filas duplicadas.
- Estandarizar textos con inconsistencias de escritura mediante funciones de ventana y *joins*.
- Tratar valores faltantes, inválidos y atípicos con criterios explícitos.
- Derivar variables de fecha.
- Escribir el resultado en Parquet particionado y comparar su eficiencia con el formato original.

| Aspecto | Detalle |
|---|---|
| Duración estimada | 45 minutos |
| Requisito previo | Cuaderno 03 |
| Entrada | Dataset activo (versión original) |
| Salida | `data/procesado/<nombre>_limpio/` (Parquet) |

### Cómo utilizar este cuaderno

1. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con `Shift + Enter`. Cada celda puede depender de variables creadas en celdas anteriores.
2. Las celdas que comienzan con `# >>> CELDA EDITABLE` son las únicas que deben modificarse para adaptar el análisis a otro dataset. El resto del código se ajusta automáticamente a las columnas detectadas.
3. Cada sección sigue la misma estructura: **fundamento** (qué es y por qué se hace), **código**, **interpretación** del resultado y **adaptación** al dataset propio.
4. Ante un error `NameError` (variable no definida), lo habitual es que no se haya ejecutado una celda anterior. La solución es *Kernel > Restart Kernel and Run All Cells*.
5. Mientras la sesión de Spark está activa, la Spark UI (normalmente <http://localhost:4040>) muestra los *jobs*, *stages* y tareas ejecutados.

La guía completa de uso de los cuadernos se encuentra en `docs/08-uso-de-notebooks.md`.

## Fundamento: el proceso ETL

ETL (*Extract, Transform, Load*) designa el proceso de extraer datos de una fuente, transformarlos (limpieza, estandarización, enriquecimiento) y cargarlos en un formato o sistema adecuado para el análisis. Wickham (2014) propone que un conjunto de datos ordenado (*tidy*) tenga una variable por columna, una observación por fila y valores consistentes; esos son los criterios que guían la limpieza de este cuaderno.

Cada paso registra cuántas filas se conservan, para documentar el efecto de la limpieza.

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("04_Limpieza")

## Carga del dataset activo

El dataset activo es el que se configuró en el cuaderno `01_configuracion_dataset.ipynb` y quedó guardado en `data/dataset_config.json`. Si no se ha configurado ninguno, se utiliza el dataset de ejemplo (`data/ejemplo/ventas_tienda.csv`).

La función `clasificar_columnas()` asigna a cada columna un rol, que los cuadernos usan para decidir qué análisis aplicar:

| Variable | Rol | Criterio |
|---|---|---|
| `NUM` | numéricas | tipos numéricos que no son identificadores |
| `CAT` | categóricas | texto o booleanos con pocos valores distintos (hasta `max_categorias`) |
| `FEC` | fechas | tipos `date` o `timestamp` |
| `TXT` | texto | texto con muchos valores distintos (comentarios, descripciones) |
| — | identificadores | columnas excluidas en la configuración o con valores casi únicos |

Con un dataset propio, conviene revisar esta clasificación. Si una columna quedó en un rol inadecuado, puede corregirse en la configuración (`columnas_excluir`, `max_categorias`) o directamente en las listas, por ejemplo `CAT.remove("columna")`.

In [ ]:
# Carga del dataset activo (definido y guardado en el cuaderno 01)
df, cfg = cargar_desde_config(spark, version="original")

# Clasificación automática de columnas por rol analítico
roles = clasificar_columnas(df, excluir=cfg["columnas_excluir"], max_categorias=cfg["max_categorias"])
NUM, CAT, FEC, TXT = roles["numericas"], roles["categoricas"], roles["fechas"], roles["texto"]
objetivo = columna_objetivo(cfg, df)
resumen_roles(roles)
print("Columna objetivo:", objetivo)

In [ ]:
import time
import pandas as pd

bitacora = []                                   # registro de cada paso de limpieza
def registrar(paso, datos):
    n = datos.count()
    bitacora.append((paso, n))
    print(f"{paso:<40}{n:>10,} filas")
    return datos

df0 = registrar("0. Datos originales", df)

## 1. Filas duplicadas

`dropDuplicates()` elimina las filas idénticas en todas las columnas. Si se le entrega una lista de columnas, por ejemplo `dropDuplicates(["id_venta"])`, considera duplicadas las filas que coinciden solo en esas columnas.

In [ ]:
# >>> CELDA EDITABLE: columnas que identifican un registro único (None = todas las columnas)
CLAVE_DUPLICADOS = None

In [ ]:
df1 = df0.dropDuplicates(CLAVE_DUPLICADOS) if CLAVE_DUPLICADOS else df0.dropDuplicates()
df1 = registrar("1. Sin duplicados", df1)

## 2. Estandarización de textos

Los textos con espacios sobrantes o mayúsculas inconsistentes (`"Bogotá"`, `" Bogotá"`, `"BOGOTÁ"`) se cuentan como categorías distintas. La estrategia aplicada tiene dos pasos:

1. `trim` y `regexp_replace` eliminan los espacios al inicio, al final y repetidos.
2. Para cada valor escrito de distintas formas, se conserva la **variante más frecuente**. Para ello se agrupa por la versión en minúsculas, se ordenan las variantes con una **función de ventana** (`row_number` sobre `Window.partitionBy`) y el resultado se une al dataset con un **join**.

Esta estrategia no inventa nombres nuevos (como haría convertir todo a mayúsculas) y funciona con cualquier columna de texto.

In [ ]:
from pyspark.sql import Window

columnas_texto = [c for c, t in df1.dtypes if t == "string"]
df2 = df1
for c in columnas_texto:
    df2 = df2.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))

def unificar_variantes(datos, c):
    variantes = (datos.where(F.col(c).isNotNull())
                 .groupBy(F.lower(F.col(c)).alias("_clave"), F.col(c).alias("_variante")).count())
    ventana = Window.partitionBy("_clave").orderBy(F.desc("count"), F.col("_variante"))
    canonica = (variantes.withColumn("_orden", F.row_number().over(ventana))
                .where("_orden = 1").select("_clave", F.col("_variante").alias("_canonica")))
    return (datos.withColumn("_clave", F.lower(F.col(c)))
                 .join(F.broadcast(canonica), on="_clave", how="left")
                 .withColumn(c, F.coalesce(F.col("_canonica"), F.col(c)))
                 .drop("_clave", "_canonica"))

antes = {c: df1.select(c).distinct().count() for c in CAT if c in columnas_texto}
for c in antes:
    df2 = unificar_variantes(df2, c)
for c, n in antes.items():
    print(f"{c:<25} valores distintos: {n:>4} -> {df2.select(c).distinct().count():>4}")

**Interpretación.** En el dataset de ejemplo, `ciudad` pasa de alrededor de 45 valores distintos a 12, que corresponde al número real de ciudades. El uso de `F.broadcast()` indica a Spark que envíe la tabla pequeña de equivalencias a todos los *executors*, de modo que el *join* no requiere *shuffle* del dataset grande (Damji et al., 2020).

## 3. Valores faltantes

Existen tres estrategias habituales:

| Estrategia | Cuándo usarla |
|---|---|
| Eliminar la fila | El valor es indispensable (por ejemplo, la variable objetivo) |
| Imputar la mediana (numéricas) | Pocos faltantes y distribución asimétrica; la mediana es robusta a atípicos |
| Imputar una categoría `Desconocido` (texto) | Conserva la fila y hace explícito el faltante |

Por defecto se eliminan las filas sin valor en la variable objetivo y se imputan las demás.

In [ ]:
print("Nulos por columna antes de tratar:")
df2.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df2.columns]).show(vertical=True)

In [ ]:
# >>> CELDA EDITABLE: estrategia de valores faltantes
ELIMINAR_SI_NULO = [objetivo] if objetivo else []   # columnas indispensables
IMPUTAR_NUMERICAS = "mediana"                      # "mediana", "media" o None
IMPUTAR_CATEGORICAS = "Desconocido"                # texto de reemplazo o None

In [ ]:
df3 = df2.dropna(subset=ELIMINAR_SI_NULO) if ELIMINAR_SI_NULO else df2

num_imputar = [c for c in NUM if c not in ELIMINAR_SI_NULO]
if IMPUTAR_NUMERICAS and num_imputar:
    if IMPUTAR_NUMERICAS == "media":
        valores = df3.agg(*[F.avg(c).alias(c) for c in num_imputar]).first().asDict()
    else:
        valores = dict(zip(num_imputar, [q[0] if q else None for q in df3.approxQuantile(num_imputar, [0.5], 0.01)]))
    valores = {c: v for c, v in valores.items() if v is not None}
    print("Valores de imputación:", {c: round(v, 2) for c, v in valores.items()})
    df3 = df3.fillna(valores)

cat_texto = [c for c in CAT if dict(df3.dtypes)[c] == "string"]
if IMPUTAR_CATEGORICAS and cat_texto:
    df3 = df3.fillna(IMPUTAR_CATEGORICAS, subset=cat_texto)

df3 = registrar("3. Tratamiento de faltantes", df3)

## 4. Valores inválidos

Un valor es inválido cuando contradice una regla del negocio, por ejemplo una cantidad negativa o una edad de 300 años. Estas reglas dependen del dominio y no pueden deducirse automáticamente; por eso se definen en una celda editable. La celda siguiente informa qué columnas numéricas contienen valores negativos, como orientación.

In [ ]:
if NUM:
    negativos = df3.agg(*[F.sum((F.col(c) < 0).cast("int")).alias(c) for c in NUM]).first().asDict()
    print("Registros con valores negativos por columna:", {c: n for c, n in negativos.items() if n})

In [ ]:
# >>> CELDA EDITABLE: reglas de validez
NO_NEGATIVAS = [c for c in ["cantidad", "precio_unitario", "total"] if c in df3.columns]   # ejemplo
RANGOS = {}            # p. ej. {"edad_cliente": (0, 110)}  -> se conservan los valores dentro del rango

In [ ]:
condicion = F.lit(True)
for c in NO_NEGATIVAS:
    condicion = condicion & (F.col(c) >= 0)
for c, (minimo, maximo) in RANGOS.items():
    condicion = condicion & F.col(c).between(minimo, maximo)
df4 = registrar("4. Reglas de validez", df3.filter(condicion))

## 5. Valores atípicos

Se aplica el criterio de Tukey (1977): un valor es atípico si está por debajo de Q1 − 1,5·RIC o por encima de Q3 + 1,5·RIC, donde RIC = Q3 − Q1. Un atípico no es necesariamente un error (por ejemplo, una compra grande legítima); por defecto solo se reportan.

In [ ]:
# >>> CELDA EDITABLE
ELIMINAR_ATIPICOS_EN = []      # columnas en las que se eliminarán los atípicos, p. ej. ["total"]

In [ ]:
limites = {}
if NUM:
    for c, (q1, q3) in zip(NUM, df4.approxQuantile(NUM, [0.25, 0.75], 0.01)):
        ric = q3 - q1
        limites[c] = (q1 - 1.5 * ric, q3 + 1.5 * ric)
    conteos = df4.agg(*[F.sum((~F.col(c).between(*limites[c])).cast("int")).alias(c) for c in NUM]).first().asDict()
    print(pd.DataFrame({"limite_inferior": {c: l[0] for c, l in limites.items()},
                        "limite_superior": {c: l[1] for c, l in limites.items()},
                        "atipicos": conteos}).round(2).to_string())

df5 = df4
for c in ELIMINAR_ATIPICOS_EN:
    df5 = df5.filter(F.col(c).between(*limites[c]))
df5 = registrar("5. Tratamiento de atípicos", df5)

## 6. Variables derivadas de la fecha

A partir de la fecha principal se crean año, trimestre, mes y día de la semana (1 = domingo, 7 = sábado). Estas columnas facilitan los análisis por periodo y sirven como variables del modelo en el cuaderno 06. La columna `anio` se usa además para particionar el archivo Parquet.

In [ ]:
col_fecha = normalizar_nombre(cfg["columna_fecha"]) if cfg.get("columna_fecha") else None
col_fecha = col_fecha if col_fecha in FEC else (FEC[0] if FEC else None)

df6 = df5
if col_fecha:
    df6 = (df5.withColumn("anio", F.year(col_fecha))
              .withColumn("trimestre", F.quarter(col_fecha))
              .withColumn("mes", F.month(col_fecha))
              .withColumn("dia_semana", F.dayofweek(col_fecha)))
    df6.select(col_fecha, "anio", "trimestre", "mes", "dia_semana").show(5)
else:
    print("Sin columna de fecha: no se derivan variables temporales.")

## 7. Enriquecimiento mediante join (demostración)

Un *join* combina dos DataFrames por una o más columnas comunes. Como ejemplo aplicable a cualquier dataset, se construye una tabla de referencia con el promedio de la variable numérica principal por categoría y se une al dataset para calcular cuánto se aparta cada registro del promedio de su grupo.

Esta columna se usa solo para ilustrar el *join* y **no se guarda** en la versión limpia: al derivarse de la variable objetivo, introduciría información del resultado en el modelo (fuga de datos).

In [ ]:
if CAT and NUM:
    n1 = objetivo if objetivo in NUM else NUM[0]
    referencia = df6.groupBy(CAT[0]).agg(F.avg(n1).cast("decimal(22,2)").alias(f"promedio_{n1}_grupo"))
    referencia.show(5)
    (df6.join(F.broadcast(referencia), on=CAT[0], how="left")
        .withColumn("diferencia_vs_grupo", (F.col(n1) - F.col(f"promedio_{n1}_grupo")).cast("decimal(22,2)"))
        .select(CAT[0], n1, f"promedio_{n1}_grupo", "diferencia_vs_grupo")
        .show(5))

## 8. Escritura en Parquet

**Parquet** es un formato columnar: almacena juntos los valores de cada columna, los comprime de forma eficiente y guarda el esquema y estadísticas por bloque. Una consulta que necesita pocas columnas solo lee esas columnas, y los filtros pueden omitir bloques completos (The Apache Software Foundation, s.f.).

Con `partitionBy("anio")`, Spark crea una subcarpeta por año (`anio=2025/`). Al filtrar por año, solo se leen las carpetas necesarias (*partition pruning*).

In [ ]:
nombre = cfg["nombre"]
ruta_salida = f"data/procesado/{nombre}_limpio"
particion = ["anio"] if "anio" in df6.columns and df6.select("anio").distinct().count() <= 50 else []

escritor = df6.write.mode("overwrite")
if particion:
    escritor = escritor.partitionBy(*particion)
escritor.parquet(str(ruta_absoluta(ruta_salida)))
print("Escrito en:", ruta_salida, "| partición:", particion or "ninguna")

cfg["ruta_limpia"] = ruta_salida
guardar_config(cfg)

## 9. Resumen del proceso y comparación de formatos

In [ ]:
resumen = pd.DataFrame(bitacora, columns=["paso", "filas"])
resumen["filas_eliminadas"] = resumen["filas"].shift(1, fill_value=resumen["filas"][0]) - resumen["filas"]
resumen

In [ ]:
print(f"Tamaño del archivo original : {tamano_mb(cfg['ruta']):8.2f} MB")
print(f"Tamaño en Parquet (limpio)  : {tamano_mb(ruta_salida):8.2f} MB")

limpio = spark.read.parquet(str(ruta_absoluta(ruta_salida)))
if NUM:
    t = time.time()
    df.groupBy(*CAT[:1]).agg(F.sum(NUM[0])).collect()
    t_original = time.time() - t
    t = time.time()
    limpio.groupBy(*CAT[:1]).agg(F.sum(NUM[0])).collect()
    print(f"Agregación sobre el original: {t_original:.2f} s | sobre Parquet: {time.time() - t:.2f} s")

**Interpretación.** Parquet suele ocupar una fracción del tamaño de un CSV equivalente. La diferencia de tiempo es pequeña con pocos datos; con millones de filas (`python scripts/generar_datos.py --ventas 2000000`) la ventaja de Parquet es notable.

### Poda de particiones

En el plan de ejecución, la línea `PartitionFilters` confirma que Spark solo leerá la carpeta del año filtrado.

In [ ]:
if particion:
    anio = limpio.select("anio").first()[0]
    limpio.filter(F.col("anio") == anio).explain()

## 10. Ejecución del mismo proceso con spark-submit

El programa `src/etl_generico.py` aplica una versión resumida de esta limpieza (duplicados, espacios, faltantes de la variable objetivo y variables de fecha) sobre el dataset configurado. Se ejecuta desde la raíz del repositorio:

```bash
spark-submit src/etl_generico.py
```

Esto ilustra el paso de un análisis interactivo a un trabajo programado, que puede ejecutarse en un clúster sin intervención manual.

## Ejercicios propuestos

1. Definir en `RANGOS` una regla de validez adecuada para el dataset propio y reportar cuántas filas elimina.
2. Comparar el número de filas finales al imputar con la media y con la mediana. ¿Cambian los resultados del análisis?
3. Escribir el dataset limpio particionado por una columna categórica y verificar la poda de particiones con `explain()`.
4. Documentar en una tabla Markdown cada decisión de limpieza tomada y su justificación.

In [ ]:
spark.stop()

## Referencias

Damji, J. S., Wenig, B., Das, T., & Lee, D. (2020). *Learning Spark: Lightning-fast data analytics* (2.ª ed.). O'Reilly Media.

The Apache Software Foundation. (s.f.). *Apache Parquet documentation* [Documentación técnica]. Recuperado el 9 de noviembre de 2026, de https://parquet.apache.org/docs/

Tukey, J. W. (1977). *Exploratory data analysis*. Addison-Wesley.

Wickham, H. (2014). Tidy data. *Journal of Statistical Software, 59*(10), 1–23. https://doi.org/10.18637/jss.v059.i10